In [34]:
import pandas as pd
import re

pd.set_option('display.max_columns', None)

In [35]:
file_path = 'raw_data/adsc_decoded.txt'

# Initialize empty containers and state variables
records = []
current_record = {}
current_tag = None
nested_prefix = ""

# Pre-compile the regular expression for the flight header line to speed up the loop
header_regex = re.compile(r"Registration:\s*(\S+)\s+ICAO ID:\s*(\S+)\s+ATSU Adress:\s*(.+)")

In [36]:
print("Parsing text file...")

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        line_stripped = line.strip()

        # Blank line signifies the end of a message block
        if not line_stripped:
            if current_record:
                records.append(current_record)
                current_record = {}
                current_tag = None
                nested_prefix = ""
            continue

        # 1. Parse Flight Information Header
        if line_stripped.startswith("Registration:"):
            match = header_regex.search(line_stripped)
            if match:
                current_record['Registration'] = match.group(1)
                current_record['ICAO_ID'] = match.group(2)
                current_record['ATSU_Address'] = match.group(3)

        elif line_stripped.startswith("Channel Frequency:"):
            current_record['Channel_Frequency'] = line_stripped.split(":", 1)[1].strip()

        elif line_stripped.startswith("/"):
            current_record['Raw_Message'] = line_stripped

        # 2. Parse Tags
        elif line_stripped.startswith("Tag"):
            tag_match = re.match(r"(Tag \d+ [^:]+):\s*(.*)", line_stripped)
            if tag_match:
                current_tag = tag_match.group(1).replace(" ", "_")
                current_record[f'{current_tag}_Hex'] = tag_match.group(2)
                nested_prefix = ""

        # 3. Parse CRC footer
        elif line_stripped.startswith("CRC:"):
            current_record['CRC'] = line_stripped.split(":", 1)[1].strip()

        # 4. Parse Tag Data (Key-Value pairs)
        else:
            if line_stripped.endswith(":"):
                nested_prefix = line_stripped[:-1].strip().replace(" ", "_") + "_"
            elif ":" in line_stripped:
                key, val = line_stripped.split(":", 1)
                key = key.strip().replace(" ", "_")
                val = val.strip()

                # Clean units so the data can be parsed as numeric later
                units_to_strip = [" ft/min", " ft", " nm", " deg", " kt", " C", "<"]
                for unit in units_to_strip:
                    val = val.replace(unit, "")

                col_name = f"{current_tag}_{nested_prefix}{key}"
                current_record[col_name] = val

    # Append the final record in case the file doesn't end with a blank line
    if current_record:
        records.append(current_record)

print(f"Successfully extracted {len(records)} message blocks into memory.")

Parsing text file...
Successfully extracted 227126 message blocks into memory.


In [37]:
df = pd.DataFrame(records)

numeric_keywords = ['Lat', 'Lon', 'Alt', 'speed', 'direction', 'Temperature', 'accuracy']

for col in df.columns:
    # Ignore hex strings, only convert actual data columns
    if any(kw in col for kw in numeric_keywords) and 'Hex' not in col:
        try:
            df[col] = pd.to_numeric(df[col])
        except ValueError:
            pass # Failsafe in case a column contains unexpected non-numeric text

print("Data successfully converted to a Pandas DataFrame.")

Data successfully converted to a Pandas DataFrame.


In [38]:
df = df.drop(columns=['Raw_Message'])

In [39]:
df = df.drop(columns=[col for col in df.columns if 'Hex' in col])
df.head()

,Registration,ICAO_ID,ATSU_Address,Channel_Frequency,Tag_07_Basic_report_Latitude,Tag_07_Basic_report_Longitude,Tag_07_Basic_report_Altitude,Tag_07_Basic_report_Timestamp,Tag_07_Basic_report_Position_accuracy,Tag_07_Basic_report_NAV_redundancy,Tag_07_Basic_report_TCAS,Tag_13_Predicted_Route_Next_waypoint_Lat,Tag_13_Predicted_Route_Next_waypoint_Lon,Tag_13_Predicted_Route_Next_waypoint_Alt,Tag_13_Predicted_Route_Next_waypoint_ETA,Tag_13_Predicted_Route_Next+1_waypoint_Lat,Tag_13_Predicted_Route_Next+1_waypoint_Lon,Tag_13_Predicted_Route_Next+1_waypoint_Alt,CRC,Tag_12_Flight_ID_Flight_ID,Tag_16_Meteorological_data_Wind_speed,Tag_16_Meteorological_data_True_wind_direction,Tag_16_Meteorological_data_Temperature,Tag_03_Acknowledgement_Contract_Number,Tag_14_Earth_Reference_data_True_track,Tag_14_Earth_Reference_data_Ground_speed,Tag_14_Earth_Reference_data_Vertical_speed,Tag_20_Waypoint_Change_Event_Latitude,Tag_20_Waypoint_Change_Event_Longitude,Tag_20_Waypoint_Change_Event_Altitude,Tag_20_Waypoint_Change_Event_Timestamp,Tag_20_Waypoint_Change_Event_Position_accuracy,Tag_20_Waypoint_Change_Event_NAV_redundancy,Tag_20_Waypoint_Change_Event_TCAS,Tag_14_Air_Reference_Data_True_heading,Tag_14_Air_Reference_Data_Mach_speed,Tag_14_Air_Reference_Data_Vertical_speed,Tag_19_Altitude_Range_Event_Latitude,Tag_19_Altitude_Range_Event_Longitude,Tag_19_Altitude_Range_Event_Altitude,Tag_19_Altitude_Range_Event_Timestamp,Tag_19_Altitude_Range_Event_Position_accuracy,Tag_19_Altitude_Range_Event_NAV_redundancy,Tag_19_Altitude_Range_Event_TCAS,Tag_05_Noncompliance_Notification_Contract_Number,Tag_23_Fixed_Projected_Intent_Group_Lat,Tag_23_Fixed_Projected_Intent_Group_Lon,Tag_23_Fixed_Projected_Intent_Group_Alt,Tag_23_Fixed_Projected_Intent_Group_ETA,Tag_22_Intermediate_Projected_Intent_Group_Distance,Tag_22_Intermediate_Projected_Intent_Group_True_track,Tag_22_Intermediate_Projected_Intent_Group_Altitude,Tag_22_Intermediate_Projected_Intent_Group_ETA,Tag_04_Negative_Acknowledgement_Contract_Request_Number,Tag_04_Negative_Acknowledgement_Reason,Tag_18_Vertical_Rate_Change_Event_Latitude,Tag_18_Vertical_Rate_Change_Event_Longitude,Tag_18_Vertical_Rate_Change_Event_Altitude,Tag_18_Vertical_Rate_Change_Event_Timestamp,Tag_18_Vertical_Rate_Change_Event_Position_accuracy,Tag_18_Vertical_Rate_Change_Event_NAV_redundancy,Tag_18_Vertical_Rate_Change_Event_TCAS,Tag_10_Lateral_Deviation_Change_Event_Latitude,Tag_10_Lateral_Deviation_Change_Event_Longitude,Tag_10_Lateral_Deviation_Change_Event_Altitude,Tag_10_Lateral_Deviation_Change_Event_Timestamp,Tag_10_Lateral_Deviation_Change_Event_Position_accuracy,Tag_10_Lateral_Deviation_Change_Event_NAV_redundancy,Tag_10_Lateral_Deviation_Change_Event_TCAS,Tag_09_Emergency_Basic_report_Latitude,Tag_09_Emergency_Basic_report_Longitude,Tag_09_Emergency_Basic_report_Altitude,Tag_09_Emergency_Basic_report_Timestamp,Tag_09_Emergency_Basic_report_Position_accuracy,Tag_09_Emergency_Basic_report_NAV_redundancy,Tag_09_Emergency_Basic_report_TCAS,Tag_17_Airframe_Identification_Group_ICAO_ID
0,A7-ANR,06a12e,BDOCAYA - Bodo Oceanic,1526377000,77.837105,35.435715,35000.0,2023-07-07 15:07:11.000,0.05,OK,OK,77.999840,34.999866,35000.0,2023-07-07 15:08:42.000,80.016689,30.164852,35000.0,A626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,N790AN,aab812,YQXE2YA - Gander Oceanic,1526377000,51.394043,-33.619194,38000.0,2023-07-07 15:07:16.250,0.25,OK,OK,49.999981,-40.000019,38000.0,2023-07-07 15:40:20.250,47.999954,-49.999981,38000.0,48F8,AAL199,85.0,343.828125,-59.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PH-BKC,485b43,PIKCPYA - Shanwick,1526377000,50.665340,-13.524342,33992.0,202

In [40]:
df['Tag_07_Basic_report_NAV_redundancy'].value_counts()

Tag_07_Basic_report_NAV_redundancy
OK      145512
lost       950
Name: count, dtype: int64

In [41]:
mapping = {
    'OK': 0,
    'lost': 1
}

# Values not in the map ('C') become NaN
df['Tag_07_Basic_report_NAV_redundancy'] = df['Tag_07_Basic_report_NAV_redundancy'].map(mapping)

In [42]:
df['Tag_07_Basic_report_NAV_redundancy'].value_counts()

Tag_07_Basic_report_NAV_redundancy
0.0    145512
1.0       950
Name: count, dtype: int64

In [43]:
df['Tag_07_Basic_report_TCAS'].value_counts()

Tag_07_Basic_report_TCAS
OK                      143650
not available to ADS      2812
Name: count, dtype: int64

In [44]:
mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_07_Basic_report_TCAS'] = df['Tag_07_Basic_report_TCAS'].map(mapping)

In [45]:
df['Tag_07_Basic_report_TCAS'].value_counts()

Tag_07_Basic_report_TCAS
0.0    143650
1.0      2812
Name: count, dtype: int64

In [47]:
df['Tag_13_Predicted_Route_Next_waypoint_ETA'] = pd.to_datetime(df['Tag_13_Predicted_Route_Next_waypoint_ETA'], errors='coerce')


In [52]:
# 2. Everything following them (.*)
regex_pattern = r'^([A-Z]+)(.*)'

# Apply the split and expand into two new columns
df[['Tag_12_Airline', 'Tag_12_Flight_Number']] = df['Tag_12_Flight_ID_Flight_ID'].str.extract(regex_pattern)

In [53]:
df['Tag_20_Waypoint_Change_Event_Timestamp'] = pd.to_datetime(df['Tag_20_Waypoint_Change_Event_Timestamp'], errors='coerce')


In [54]:
df['Tag_20_Waypoint_Change_Event_NAV_redundancy'].value_counts()

Tag_20_Waypoint_Change_Event_NAV_redundancy
OK      44006
lost      118
Name: count, dtype: int64

In [55]:
mapping = {
    'OK': 0,
    'lost': 1
}

df['Tag_20_Waypoint_Change_Event_NAV_redundancy'] = df['Tag_20_Waypoint_Change_Event_NAV_redundancy'].map(mapping)

In [56]:
mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_20_Waypoint_Change_Event_TCAS'] = df['Tag_20_Waypoint_Change_Event_TCAS'].map(mapping)

In [57]:
mapping = {
    'OK': 0,
    'lost': 1
}

df['Tag_19_Altitude_Range_Event_NAV_redundancy'] = df['Tag_19_Altitude_Range_Event_NAV_redundancy'].map(mapping)

In [58]:
mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_19_Altitude_Range_Event_TCAS'] = df['Tag_19_Altitude_Range_Event_TCAS'].map(mapping)

In [59]:
df['Tag_23_Fixed_Projected_Intent_Group_ETA'] = pd.to_datetime(df['Tag_23_Fixed_Projected_Intent_Group_ETA'], errors='coerce')

In [60]:
df['Tag_22_Intermediate_Projected_Intent_Group_ETA'] = pd.to_datetime(df['Tag_22_Intermediate_Projected_Intent_Group_ETA'], errors='coerce')

In [61]:
df['Tag_18_Vertical_Rate_Change_Event_Timestamp'] = pd.to_datetime(df['Tag_18_Vertical_Rate_Change_Event_Timestamp'], errors='coerce')

In [63]:
mapping = {
    'OK': 0,
    'lost': 1
}

df['Tag_18_Vertical_Rate_Change_Event_NAV_redundancy'] = df['Tag_18_Vertical_Rate_Change_Event_NAV_redundancy'].map(mapping)

mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_18_Vertical_Rate_Change_Event_TCAS'] = df['Tag_18_Vertical_Rate_Change_Event_TCAS'].map(mapping)

In [64]:
df['Tag_10_Lateral_Deviation_Change_Event_Timestamp'] = pd.to_datetime(df['Tag_10_Lateral_Deviation_Change_Event_Timestamp'], errors='coerce')

In [65]:
mapping = {
    'OK': 0,
    'lost': 1
}

df['Tag_10_Lateral_Deviation_Change_Event_NAV_redundancy'] = df['Tag_10_Lateral_Deviation_Change_Event_NAV_redundancy'].map(mapping)

mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_10_Lateral_Deviation_Change_Event_TCAS'] = df['Tag_10_Lateral_Deviation_Change_Event_TCAS'].map(mapping)

In [66]:
df['Tag_09_Emergency_Basic_report_Timestamp'] = pd.to_datetime(df['Tag_09_Emergency_Basic_report_Timestamp'], errors='coerce')

In [67]:
mapping = {
    'OK': 0,
    'lost': 1
}

df['Tag_09_Emergency_Basic_report_NAV_redundancy'] = df['Tag_09_Emergency_Basic_report_NAV_redundancy'].map(mapping)

mapping = {
    'OK': 0,
    'not available to ADS': 1
}

# Values not in the map ('C') become NaN
df['Tag_09_Emergency_Basic_report_TCAS'] = df['Tag_09_Emergency_Basic_report_TCAS'].map(mapping)

In [70]:
df.columns

Index(['Registration', 'ICAO_ID', 'ATSU_Address', 'Channel_Frequency',
       'Tag_07_Basic_report_Latitude', 'Tag_07_Basic_report_Longitude',
       'Tag_07_Basic_report_Altitude', 'Tag_07_Basic_report_Timestamp',
       'Tag_07_Basic_report_Position_accuracy',
       'Tag_07_Basic_report_NAV_redundancy', 'Tag_07_Basic_report_TCAS',
       'Tag_13_Predicted_Route_Next_waypoint_Lat',
       'Tag_13_Predicted_Route_Next_waypoint_Lon',
       'Tag_13_Predicted_Route_Next_waypoint_Alt',
       'Tag_13_Predicted_Route_Next_waypoint_ETA',
       'Tag_13_Predicted_Route_Next+1_waypoint_Lat',
       'Tag_13_Predicted_Route_Next+1_waypoint_Lon',
       'Tag_13_Predicted_Route_Next+1_waypoint_Alt', 'CRC',
       'Tag_12_Flight_ID_Flight_ID', 'Tag_16_Meteorological_data_Wind_speed',
       'Tag_16_Meteorological_data_True_wind_direction',
       'Tag_16_Meteorological_data_Temperature',
       'Tag_03_Acknowledgement_Contract_Number',
       'Tag_14_Earth_Reference_data_True_track',
       'Tag_

In [71]:
def normalize_columns(columns):
    normalized = []
    for col in columns:
        # 1. Convert to lowercase
        # 2. Replace '+' and other non-alphanumeric chars with '_'
        # 3. Replace multiple underscores with a single one
        # 4. Strip leading/trailing underscores
        clean_name = re.sub(r'[^a-zA-Z0-9]', '_', col).lower()
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        normalized.append(clean_name)
    return normalized

# Apply normalization
df.columns = normalize_columns(df.columns)

In [72]:
df.head()

,registration,icao_id,atsu_address,channel_frequency,tag_07_basic_report_latitude,tag_07_basic_report_longitude,tag_07_basic_report_altitude,tag_07_basic_report_timestamp,tag_07_basic_report_position_accuracy,tag_07_basic_report_nav_redundancy,tag_07_basic_report_tcas,tag_13_predicted_route_next_waypoint_lat,tag_13_predicted_route_next_waypoint_lon,tag_13_predicted_route_next_waypoint_alt,tag_13_predicted_route_next_waypoint_eta,tag_13_predicted_route_next_1_waypoint_lat,tag_13_predicted_route_next_1_waypoint_lon,tag_13_predicted_route_next_1_waypoint_alt,crc,tag_12_flight_id_flight_id,tag_16_meteorological_data_wind_speed,tag_16_meteorological_data_true_wind_direction,tag_16_meteorological_data_temperature,tag_03_acknowledgement_contract_number,tag_14_earth_reference_data_true_track,tag_14_earth_reference_data_ground_speed,tag_14_earth_reference_data_vertical_speed,tag_20_waypoint_change_event_latitude,tag_20_waypoint_change_event_longitude,tag_20_waypoint_change_event_altitude,tag_20_waypoint_change_event_timestamp,tag_20_waypoint_change_event_position_accuracy,tag_20_waypoint_change_event_nav_redundancy,tag_20_waypoint_change_event_tcas,tag_14_air_reference_data_true_heading,tag_14_air_reference_data_mach_speed,tag_14_air_reference_data_vertical_speed,tag_19_altitude_range_event_latitude,tag_19_altitude_range_event_longitude,tag_19_altitude_range_event_altitude,tag_19_altitude_range_event_timestamp,tag_19_altitude_range_event_position_accuracy,tag_19_altitude_range_event_nav_redundancy,tag_19_altitude_range_event_tcas,tag_05_noncompliance_notification_contract_number,tag_23_fixed_projected_intent_group_lat,tag_23_fixed_projected_intent_group_lon,tag_23_fixed_projected_intent_group_alt,tag_23_fixed_projected_intent_group_eta,tag_22_intermediate_projected_intent_group_distance,tag_22_intermediate_projected_intent_group_true_track,tag_22_intermediate_projected_intent_group_altitude,tag_22_intermediate_projected_intent_group_eta,tag_04_negative_acknowledgement_contract_request_number,tag_04_negative_acknowledgement_reason,tag_18_vertical_rate_change_event_latitude,tag_18_vertical_rate_change_event_longitude,tag_18_vertical_rate_change_event_altitude,tag_18_vertical_rate_change_event_timestamp,tag_18_vertical_rate_change_event_position_accuracy,tag_18_vertical_rate_change_event_nav_redundancy,tag_18_vertical_rate_change_event_tcas,tag_10_lateral_deviation_change_event_latitude,tag_10_lateral_deviation_change_event_longitude,tag_10_lateral_deviation_change_event_altitude,tag_10_lateral_deviation_change_event_timestamp,tag_10_lateral_deviation_change_event_position_accuracy,tag_10_lateral_deviation_change_event_nav_redundancy,tag_10_lateral_deviation_change_event_tcas,tag_09_emergency_basic_report_latitude,tag_09_emergency_basic_report_longitude,tag_09_emergency_basic_report_altitude,tag_09_emergency_basic_report_timestamp,tag_09_emergency_basic_report_position_accuracy,tag_09_emergency_basic_report_nav_redundancy,tag_09_emergency_basic_report_tcas,tag_17_airframe_identification_group_icao_id,tag_12_airline,tag_12_flight_number
0,A7-ANR,06a12e,BDOCAYA - Bodo Oceanic,1526377000,77.837105,35.435715,35000.0,2023-07-07 15:07:11.000,0.05,0.0,0.0,77.999840,34.999866,35000.0,2023-07-07 15:08:42.000,80.016689,30.164852,35000.0,A626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,N790AN,aab812,YQXE2YA - Gander Oceanic,1526377000,51.394043,-33.619194,38000.0,2023-07-07 15:07:16.250,0.25,0.0,0.0,49.999981,-40.000019,38000.0,2023-07-07 15:40:20.250,47.999954,-49.999981,38000.0,48F8,AAL199,85.0,343.828125,-59.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,AAL,199
2,PH-BKC,485b43,PIKCPY

In [74]:
df.to_csv("clean_data/adsc.csv", index=False, na_rep='')